# Orchestrator

Примеры использования пайплайна:

In [1]:
import sys
import os
import json

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# Добавляем корень репозитория в путь, чтобы импортировать пакет app
sys.path.insert(0, os.path.abspath(".."))

from app.orchestrator.runner import run
from app.models.state import PipelineResult
from app.logger import setup_logging
from app.config import get_config
from app.generator.agent import GeneratorAgent

/Users/mishasdk/repo/mipt-secure-sql-agents/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# Можно задать переменные окружения в .env, либо перезаписать тут.
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"

# Включаем логгирование.
setup_logging()

## Запуск пайплана целиком

In [3]:
query = "Show me all users who registered in the last 30 days"

result: PipelineResult = run(query)
result

2026-05-16 19:06:46 [INFO] app.orchestrator.runner: Pipeline started | query='Show me all users who registered in the last 30 days'
2026-05-16 19:06:46 [INFO] app.orchestrator: Building graph | model=openai/gpt-4o-mini
2026-05-16 19:06:47 [INFO] app.judge: load_policy | path=/Users/mishasdk/repo/mipt-secure-sql-agents/app/judge/res/security_rules.yaml | rules=4
2026-05-16 19:06:47 [INFO] app.generator: GeneratorAgent called | query='Show me all users who registered in the last 30 days'
2026-05-16 19:06:47 [INFO] app.generator.rag: tool=retriever | action=retrieve | query='Show me all users who registered in the last 30 days' | k=8
2026-05-16 19:06:47 [INFO] app.generator.rag: tool=embeddings | action=query | model=perplexity/pplx-embed-v1-4b
2026-05-16 19:06:48 [INFO] app.generator.rag: tool=embeddings | elapsed=1.25s
2026-05-16 19:06:48 [INFO] app.generator.rag: tool=retriever | result_tables=['participant_app', 'sys_company', 'scp_sec_check_res', 'scp_project_ans', 'mler_application'

PipelineResult(final_sql="SELECT id, name, create_date FROM users WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000; /* Изменено на таблицу users */", approved=True, iterations_used=1, iteration_logs=[IterationLog(iteration=0, sql_query="SELECT id, name, create_date FROM sys_employee WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000", judge_output=JudgeOutput(verdict=<Verdict.REJECTED: 'REJECTED'>, risk_score=9.0, issues=[FoundIssue(type=<IssueType.POLICY: 'POLICY'>, severity=<Severity.HIGH: 'HIGH'>, message="Table 'sys_employee' is not in the allowed list", fix_instruction='Use only allowed tables: orders, products, users')]), timestamp=datetime.datetime(2026, 5, 16, 19, 6, 43, 370063)), IterationLog(iteration=1, sql_query="SELECT id, name, create_date FROM users WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000; /* Изменено на таблицу users */", judge_output=JudgeOutput(verdict=<Verdict.APPROVED: 'APPROVED'>, risk_score=0.0, issues=[]), timestamp=datetime.d

In [4]:
print(f"Approved: {result.approved}")
print(f"Iterations used: {result.iterations_used}")
print(f"\nFinal SQL:\n{result.final_sql}")

print("\nIteration logs:")
print(json.dumps([log.model_dump(mode="json") for log in result.iteration_logs], indent=2))

Approved: True
Iterations used: 1

Final SQL:
SELECT id, name, create_date FROM users WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000; /* Изменено на таблицу users */

Iteration logs:
[
  {
    "iteration": 0,
    "sql_query": "SELECT id, name, create_date FROM sys_employee WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000",
    "judge_output": {
      "verdict": "REJECTED",
      "risk_score": 9.0,
      "issues": [
        {
          "type": "POLICY",
          "severity": "HIGH",
          "message": "Table 'sys_employee' is not in the allowed list",
          "fix_instruction": "Use only allowed tables: orders, products, users"
        }
      ]
    },
    "timestamp": "2026-05-16T19:06:43.370063"
  },
  {
    "iteration": 1,
    "sql_query": "SELECT id, name, create_date FROM users WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000; /* \u0418\u0437\u043c\u0435\u043d\u0435\u043d\u043e \u043d\u0430 \u0442\u0430\u0431\u043b\u0438\u0446\u0443 users */",
  

## Запуск Generator

In [5]:
config = get_config()
llm = ChatOpenAI(
    base_url=config.openrouter_base_url,
    api_key=config.open_router_api_key,
    model=config.model_name,
)

generator = GeneratorAgent(llm)

state = {
    "messages": [HumanMessage(content="Show me all users who registered in the last 30 days")],
    "judge_responses": [],
    "llm_calls": 0,
    "audit_iteration": 0,
}

output = generator(state)
print(output)

2026-05-16 19:08:01 [INFO] app.generator: GeneratorAgent called | query='Show me all users who registered in the last 30 days'
2026-05-16 19:08:01 [INFO] app.generator.rag: tool=retriever | action=cache_hit | query='Show me all users who registered in the last 30 days'
2026-05-16 19:08:01 [INFO] app.generator: tool=retriever | elapsed=0.00s | tables=['participant_app', 'sys_company', 'scp_sec_check_res', 'scp_project_ans', 'mler_application', 'scp_application', 'sys_employee', 'dict_product']
2026-05-16 19:08:01 [INFO] app.generator: tool=llm | action=invoke | agent=generator
2026-05-16 19:08:03 [INFO] app.generator: tool=llm | elapsed=1.87s | tables_used=['participant_app'] | sql='SELECT id, name__ru, name, create_date FROM participant_app WHERE create_date >='


{'messages': [AIMessage(content='{\n  "sql": "SELECT id, name__ru, name, create_date FROM participant_app WHERE create_date >= CURRENT_DATE - INTERVAL \'30 days\' LIMIT 1000",\n  "tables_used": [\n    "participant_app"\n  ]\n}', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'llm_calls': 1}


Других агентов можно запускать по аналогии.